In [1]:
pip install pandas

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
import pandas as pd

# =====================================================================
# 1. LOAD THE KAGGLE DATASETS
# =====================================================================
try:
    clients_df = pd.read_csv('clients_with_fatf_ofac.csv')
    print("✓ Successfully loaded clients dataset.")
    print(f"Total records found: {len(clients_df)}\n")
except FileNotFoundError:
    print("Error: Could not find 'clients_with_fatf_ofac.csv'. Ensure it's in the same directory.")
    exit()

# Let's peek at the available columns to make sure they match expectations
print("Available columns in dataset:", list(clients_df.columns))

# =====================================================================
# 2. DEFINE THE COMPLIANCE RISK SCORING ENGINE
# =====================================================================
def calculate_compliance_risk(row):
    """
    Evaluates risk vectors for an entity and assigns a weighted score.
    Max potential score: 100 points.
    """
    risk_score = 0
    
    # Risk Vector 1: Global Sanctions Match (Absolute Dealbreaker)
    # Checks if the entity matches an international regulatory watchlist (e.g., OFAC)
    if 'sanctions_flag' in row and (row['sanctions_flag'] == 1 or row['sanctions_flag'] == True):
        risk_score += 50
        
    # Risk Vector 2: Politically Exposed Persons (PEP) Link
    # Determines if the owner holds a public office, introducing potential corruption risks
    if 'pep_flag' in row and (row['pep_flag'] == 1 or row['pep_flag'] == True):
        risk_score += 25
        
    # Risk Vector 3: Jurisdictional / Country Risk
    # High scores for countries flagged by regulators like FATF for high financial crime
    if 'country_risk_score' in row:
        # Assuming the Kaggle dataset scales country risk from 1 to 5
        risk_score += (row['country_risk_score'] * 4)  # Scales country risk up to 20 points
    elif 'country_risk_category' in row and row['country_risk_category'] == 'High':
        risk_score += 15

    # Risk Vector 4: Entity / Sector Exposure (Buffer Risk)
    # Certain industries (like Cash-Intensive Retail or Crypto) hold higher inherent risk
    if 'industry_risk_level' in row and row['industry_risk_level'] == 'High':
        risk_score += 5
        
    return min(risk_score, 100)  # Ensures the score is capped perfectly at 100

# =====================================================================
# 3. RUN THE ENGINE & CATEGORIZE RISK LEVELS
# =====================================================================
# Apply the analytical framework across every row in the dataset
clients_df['Calculated_Risk_Score'] = clients_df.apply(calculate_compliance_risk, axis=1)

# Categorize risk into actionable operational brackets for stakeholders
def segment_risk_tier(score):
    if score >= 65:
        return 'High Risk (Strict Escalate / Block)'
    elif score >= 35:
        return 'Medium Risk (Enhanced Due Diligence Required)'
    else:
        return 'Low Risk (Green-light Onboarding)'

clients_df['Risk_Tier'] = clients_df['Calculated_Risk_Score'].apply(segment_risk_tier)

# =====================================================================
# 4. EXPORT COMPLIANCE-READY DATASET
# =====================================================================
output_filename = 'final_compliance_risk_analysis.csv'
clients_df.to_csv(output_filename, index=False)

print(f"\n✓ Analysis Complete!")
print(f"Processed file saved as: '{output_filename}'")
print("\nRisk Tier Distribution Summary:")
print(clients_df['Risk_Tier'].value_counts())

✓ Successfully loaded clients dataset.
Total records found: 2000

Available columns in dataset: ['client_id', 'client_name', 'client_type', 'sector', 'sector_risk', 'country', 'pep_flag', 'sanctions_flag', 'fatf_country_flag', 'ofac_country_flag', 'sectoral_sanctions_flag', 'ownership_opacity_score']

✓ Analysis Complete!
Processed file saved as: 'final_compliance_risk_analysis.csv'

Risk Tier Distribution Summary:
Risk_Tier
Low Risk (Green-light Onboarding)                1945
Medium Risk (Enhanced Due Diligence Required)      50
High Risk (Strict Escalate / Block)                 5
Name: count, dtype: int64
